In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import os

from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:
def generate_batches(text, seq_length, batch_size, unique_chars):
    encoded_text = np.array([unique_chars.index(c) for c in text])
    #coverting entire text to corr index
    num_batches = (len(encoded_text) - 1) // (batch_size * seq_length)   #if batch_size 2, and seq= ab, then 2*2=4 chars grouped together
    #-1 because we are going to make imput,target pairs and last char won't get a target
    #input, target isn't necessarily one char. seq_length specifies how many chars to take as input/target
    
    input_data = encoded_text[:num_batches * batch_size * seq_length]
    target_data = encoded_text[1:num_batches * batch_size * seq_length + 1] #shift by one char
    
    input_batches = input_data.reshape((batch_size, -1))
    target_batches = target_data.reshape((batch_size, -1)) #rows acc to batch_size
    
    for n in range(0, input_batches.shape[1], seq_length):
        x = input_batches[:, n:n+seq_length]
        y = target_batches[:, n:n+seq_length]

        #eg- seq_length=4, input- HELL, target- ELLO
        
        yield torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)
        #yield gives one sample at a time (batch)

In [3]:
class LSTM(nn.Module):
    def __init__(self, unique_chars, hidden_units=256, num_layers=3, dropout_probability=0.5, use_gpu=True):
        super().__init__()
        self.hidden_units = hidden_units
        self.num_layers = num_layers
        self.dropout_prob = dropout_probability
        self.use_gpu = use_gpu
        self.device = torch.device("cuda" if torch.cuda.is_available() and use_gpu else "cpu")

        self.encoder = {char: idx for idx, char in enumerate(unique_chars)} #dict integers to chars
        self.decoder = {idx: char for char, idx in self.encoder.items()} #dict chars to int

        self.embedding = nn.Embedding(len(unique_chars), hidden_units)  # Embedding 256 dimensional
        self.lstm = nn.LSTM(hidden_units, hidden_units, num_layers, dropout=dropout_probability, batch_first=True)
        self.dropout = nn.Dropout(self.dropout_prob)
        self.linear = nn.Linear(self.hidden_units, len(unique_chars)) #converts 256 outputs to number of unique char outputs
        #raw logits over which we will apply softmax later

        

        
        self.to(self.device)

    def forward(self, seq, hidden):    #seq is current batch, hiddent is a tuple of (h_t-1, c_t-1)
        seq = seq.to(self.device)
        embedded = self.embedding(seq) 
        lstm_output, hidden = self.lstm(embedded, hidden) #giving lstm our embedded text and hidden state
        #lstm_output stores all hidden states, hidden stores last hidden state
        
        lstm_output = self.dropout(lstm_output)
        
        lstm_output = lstm_output.contiguous().view(-1, self.hidden_units)
        #Converts shape from (batch_size, seq_length, hidden_units) → (batch_size * seq_length, hidden_units)
        
        final_output = self.linear(lstm_output)
        return final_output, hidden

    def init_hidden(self, batch_size):
        return (
            torch.zeros(self.num_layers, batch_size, self.hidden_units, device=self.device),  #empty tensors for c_t-1, h_t-1
            torch.zeros(self.num_layers, batch_size, self.hidden_units, device=self.device)   #given shape
        ) #for each batch, we need fresh 0 tensors

In [4]:
with open("/kaggle/input/shakespeare/shakespeare.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

unique_chars = sorted(set(text_data))
split_idx = int(len(text_data) * 0.9)
train_text = text_data[:split_idx]
val_text = text_data[split_idx:]


In [5]:
print(len(text_data))
unique_char = set(text_data)
print(len(unique_chars))

1115394
65


In [6]:
print(f"Train size: {len(train_text)}, Val size: {len(val_text)}, Unique chars: {len(unique_chars)}")


Train size: 1003854, Val size: 111540, Unique chars: 65


In [ ]:
seq_length = 100   #size of training example
batch_size = 64   
num_epochs = 50
learning_rate = 0.001

model = LSTM(unique_chars, hidden_units=512, num_layers=3, dropout_probability=0.5, use_gpu=True)

criterion = nn.CrossEntropyLoss()  #automatically will apply softmax
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training Loop

for epoch in range(num_epochs):
    model.train()
    hidden = model.init_hidden(batch_size)   #initializing zero tensors for hidden_state
    train_loss = 0

    for x_batch, y_batch in generate_batches(train_text, seq_length, batch_size, unique_chars):
        model.train()
        
        hidden = tuple(h.detach() for h in hidden)  # Detach hidden states- we can't reinitialize as 0 or else how will LTM store long term stuff
        #detach means keep the previous data but no back prop on it
        
        model.zero_grad()
        
        output, hidden = model(x_batch, hidden)

        """
        print(f"Output shape: {output.shape}")  
        print(f"y_batch original shape: {y_batch.shape}")  
        print(f"y_batch reshaped shape: {y_batch.view(-1).shape}") 
        print("Unique values in y_batch:", torch.unique(y_batch))
        """

        #y_batch has true target char indices of each sequence
        loss = criterion(output, y_batch.view(-1).to(model.device))  #view(-1) reshapes y_batch to a long vector 1d(batch_size*seq_length), (originally 2d(batch_size, seq_length))
        loss.backward()  #compute gradients
        optimizer.step()  #update weights
        
        train_loss += loss.item()

    # Validation
    val_loss = 0
    model.eval()
    hidden = model.init_hidden(batch_size)
    with torch.no_grad():
        for x_val, y_val in generate_batches(val_text, seq_length, batch_size, unique_chars):
            model.eval()
            output, hidden = model(x_val, hidden)
            val_loss += criterion(output, y_val.view(-1).to(model.device)).item()

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")


Epoch 1/100, Train Loss: 384.8998, Val Loss: 33.4395
Epoch 2/100, Train Loss: 286.3504, Val Loss: 29.9987
Epoch 3/100, Train Loss: 259.2484, Val Loss: 28.2893
Epoch 4/100, Train Loss: 245.1904, Val Loss: 27.3951
Epoch 5/100, Train Loss: 236.1721, Val Loss: 26.6608
Epoch 6/100, Train Loss: 229.3352, Val Loss: 26.1888
Epoch 7/100, Train Loss: 224.1358, Val Loss: 25.9240
Epoch 8/100, Train Loss: 220.2117, Val Loss: 25.6091
Epoch 9/100, Train Loss: 216.8506, Val Loss: 25.4493
Epoch 10/100, Train Loss: 213.7790, Val Loss: 25.2235
Epoch 11/100, Train Loss: 211.2111, Val Loss: 25.0714
Epoch 12/100, Train Loss: 208.8300, Val Loss: 25.0564
Epoch 13/100, Train Loss: 206.9435, Val Loss: 24.9134
Epoch 14/100, Train Loss: 204.9834, Val Loss: 24.9009
Epoch 15/100, Train Loss: 203.3645, Val Loss: 24.7622
Epoch 16/100, Train Loss: 201.9677, Val Loss: 24.7269
Epoch 17/100, Train Loss: 200.5225, Val Loss: 24.7516
Epoch 18/100, Train Loss: 199.2827, Val Loss: 24.6136
Epoch 19/100, Train Loss: 197.9466, V

In [ ]:
import torch



def generate_text(model, start_text, length, temperature=1.0):
#length how many chars to generate
    
    model.eval()
    hidden = model.init_hidden(1) # batch size=1, since we are only testing one sequence at a time, we can generate 1 character at a time
    generated_text = start_text

    input_seq = torch.tensor([model.encoder[ch] for ch in start_text], dtype=torch.long).unsqueeze(0).to(model.device)
    #shape = (1, len(start_text))

    for _ in range(length):
        with torch.no_grad():
            output, hidden = model(input_seq, hidden)

        output_dist = torch.nn.functional.softmax(output[-1] / temperature, dim=0).cpu().numpy() #-1 cause last char, one char at a time considered
        #temp controls randomness, parameter
        #dim=0 applies softmax to all axis, all outputs, convert to numpy
        #cpu as numpy doesn't support cuda
        
        next_char_idx = np.random.choice(len(model.unique_chars), p=output_dist)
        next_char = model.decoder[next_char_idx]

      
        generated_text += next_char

        input_seq = torch.tensor([[next_char_idx]], dtype=torch.long).to(model.device) #updating input

    return generated_text



In [ ]:
start = "To be or not to be"
generated_output = generate_text(model, start, length=500, temperature=0.7)
print(generated_output)